# Main experiment: chains
**Do activation edits act like seeing or like doing?**  Model: Qwen3-4B-Base. Runtime: **T4 GPU**.

Everything is read from and saved to **My Drive / see-vs-do**. Use **Runtime → Run all**. Every stage saves its results, so if Colab disconnects you can rerun from the stage that failed.

| Stage | What it does | Time on a T4 |
|---|---|---|
| 0 | Collider check: does the model explain away in text? (go / no-go for H3) | ~3 min |
| A | Model's own see / do behaviour on the held-out stories, plus the story filters | ~3 min |
| B | Concept directions per scenario and layer, checked with a probe | ~3 min |
| C | Edit sweep: real direction, random direction, unrelated direction | ~30–45 min |
| D | Do-likeness score D, heatmaps, control comparison | <1 min |

In [ ]:
# Mount Google Drive and work inside the see-vs-do folder (story files live there; results are saved there)
import os
from google.colab import drive
drive.mount("/content/drive")
os.chdir("/content/drive/MyDrive/see-vs-do")
OUT = "results"
os.makedirs(OUT, exist_ok=True)
print("working in", os.getcwd(), "| files:", sorted(os.listdir(".")))

In [ ]:
# ---------- Setup: model, tokenizer, settings
import gc, os, time, torch, numpy as np, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_grad_enabled(False)

NAME = "Qwen/Qwen3-4B-Base"
DEVICE = "cuda"
BS = 16                         # batch size; lower to 8 if you get out-of-memory errors
TRAIN_VARIANTS = range(0, 16)   # used only to learn directions
EVAL_VARIANTS = range(16, 20)   # held out; used for every score. Raise to range(16, 24) for the final run
LAYER_STEP = 4                  # plan of record: every 4th layer
ALPHAS = [1.0, 2.0, 4.0]        # edit strength, in units of the (true - false) difference vector
POSITIONS = ["mediator", "after", "answer"]

for v in ["lm", "model", "W32"]:
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load(dtype):
    m = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map=DEVICE).eval()
    return m, m.get_output_embeddings().weight.float()     # float32 output layer

lm, W32 = load(torch.float16)
N_LAYERS = lm.config.num_hidden_layers
LAYERS = list(range(0, N_LAYERS, LAYER_STEP))
print("layers to sweep:", LAYERS)

YES = tok(" Yes", add_special_tokens=False).input_ids
NO = tok(" No", add_special_tokens=False).input_ids
assert len(YES) == 1 and len(NO) == 1
YES, NO = YES[0], NO[0]

In [ ]:
# ---------- Prompt building, token positions, belief with optional edit
import ast
FEWSHOT = (
    "Story: The kettle was on the stove and the water started to bubble.\n"
    "Question: Was the water hot?\nAnswer: Yes\n\n"
    "Story: Maya left her umbrella at home and the sky stayed clear all day.\n"
    "Question: Did Maya get wet from rain?\nAnswer: No\n\n"
)
STORY_OFFSET = len(FEWSHOT) + len("Story: ")

def prompt(story, q):
    return f"{FEWSHOT}Story: {story}\nQuestion: {q}\nAnswer:"

def encode(stories, qs, spans):
    """Tokenise a batch (left-padded) and find, per row, the last token of the key sentence."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True,
              return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")
    am = enc["attention_mask"]
    key_idx = []
    for i, (a, b) in enumerate(spans):
        a, b = a + STORY_OFFSET, b + STORY_OFFSET
        hit = ((offs[i, :, 0] < b) & (offs[i, :, 1] > a) & (am[i] == 1)).nonzero().flatten()
        assert len(hit) > 0, "key sentence not found in tokens"
        key_idx.append(hit[-1].item())
    return enc, torch.tensor(key_idx)

def position_mask(key_idx, T, mode):
    B = len(key_idx)
    m = torch.zeros(B, T)
    for i, k in enumerate(key_idx.tolist()):
        if mode == "mediator":
            m[i, k] = 1
        elif mode == "after":
            m[i, k + 1:T - 1] = 1     # tokens after the key sentence, excluding the final token
        elif mode == "answer":
            m[i, T - 1] = 1
    return m

def _layer_out(out):
    return out[0] if isinstance(out, tuple) else out

def belief(stories, qs, spans, edit=None, bs=BS):
    """log P(Yes) - log P(No). edit = dict(layer=int, vecs=[N, H] tensor, mode=str) or None."""
    res = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handle = None
        if edit is not None:
            T = enc["input_ids"].shape[1]
            mask = position_mask(key_idx, T, edit["mode"]).to(DEVICE, lm.dtype)
            vecs = edit["vecs"][i:i+bs].to(DEVICE, lm.dtype)
            def hook(mod, inp, out, mask=mask, vecs=vecs):
                h = _layer_out(out) + mask[:, :, None] * vecs[:, None, :]
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handle = lm.model.layers[edit["layer"]].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            if handle is not None:
                handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

def key_states(stories, qs, spans, layers, bs=BS):
    """Residual stream (output of each layer in `layers`) at the last token of the key sentence."""
    out = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        rows = torch.arange(len(key_idx))
        out.append(torch.stack([hs[L + 1][rows, key_idx.to(DEVICE)].float().cpu() for L in layers], 1))
    return torch.cat(out)        # [N, len(layers), H]

# float16 overflow check; fall back to bfloat16 if needed
t = belief(["Priya ate a huge lunch and felt full."], ["Was Priya full?"], [[0, 5]])
if not np.isfinite(t).all():
    print("float16 overflow -> bfloat16")
    del lm, W32; gc.collect(); torch.cuda.empty_cache()
    lm, W32 = load(torch.bfloat16)
print("dtype:", lm.dtype)

d = pd.read_csv("stories_chain.csv")
d["variant"] = d.item_id.str.rsplit("_", n=1).str[1].astype(int)
for c in [c for c in d.columns if c.startswith("span_")]:
    d[c] = d[c].apply(ast.literal_eval)
train = d[d.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
ev = d[d.variant.isin(EVAL_VARIANTS)].reset_index(drop=True)
SCEN = sorted(d.scenario_id.unique())
print(f"{len(SCEN)} scenarios | train items {len(train)} | eval items {len(ev)}")

## Stage 0: collider check (go / no-go for H3)
Rule in each story: C happens if A or B. We probe belief in B.
**Explaining away** EA = belief(B | C seen) − belief(B | C seen and A stated); it should be positive.
Criteria fixed before running: a scenario is valid if seeing C raises belief in B by > 0.5 (median over its variants). **GO** if ≥ 10 valid scenarios, median EA > 0.5, and EA > 0 in ≥ 70% of them.

In [ ]:
cd_ = pd.read_csv("stories_collider.csv")
for c in [c for c in cd_.columns if c.startswith("span_")]:
    cd_[c] = cd_[c].apply(ast.literal_eval)
qb = cd_.q_b.tolist()
for cond in ["base", "see", "see_A", "A_only", "do"]:
    cd_[f"b_{cond}"] = belief(cd_[cond].tolist(), qb, cd_[f"span_{cond}"].tolist())
cd_["c_see"] = belief(cd_.see.tolist(), cd_.q_c.tolist(), cd_.span_see.tolist())
cd_["diag"] = cd_.b_see - cd_.b_base       # seeing C raises B?
cd_["EA"] = cd_.b_see - cd_.b_see_A        # explaining away
cd_["indep"] = cd_.b_A_only - cd_.b_base   # A alone should not change B
cd_["do_ref"] = cd_.b_do - cd_.b_base      # forced C should not change B
cd_.to_csv(f"{OUT}/0_collider_items.csv", index=False)
csc = cd_.groupby("scenario_id")[["diag", "EA", "indep", "do_ref", "c_see"]].median()
csc["valid"] = csc.diag > 0.5
csc.to_csv(f"{OUT}/0_collider_scenarios.csv")
display(csc.round(2).sort_values("EA", ascending=False))
cv = csc[csc.valid]
med_ea = cv.EA.median() if len(cv) else float("nan")
frac_pos = (cv.EA > 0).mean() if len(cv) else 0.0
COLLIDER_GO = (len(cv) >= 10) and (med_ea > 0.5) and (frac_pos >= 0.70)
print(f"Valid scenarios: {len(cv)}/{len(csc)} (need >= 10) | median EA {med_ea:.2f} (need > 0.5) | EA > 0 in {frac_pos:.0%} (need >= 70%)")
print(f"Independence check {cv.indep.median():.2f} (ideal ~0) | text do(C) reference {cv.do_ref.median():.2f} (ideal ~0)")
print("DECISION:", "GO - colliders stay in" if COLLIDER_GO else "NO-GO - report as a behavioural result, drop colliders from the sweeps")

## Stage A: the model's own seeing and doing (reference)
For every held-out story: how much belief in the **cause**, the **effect** and the **mediator** moves under *see* and *do*, compared with the neutral base story.
Then the story filters fixed in advance, applied per scenario:
Δcause(see) > 0.5, Δeffect(see) > 0.5, Δeffect(do) > 0.5, and r_see − r_do > 0.25, where r = Δcause / Δeffect.

In [ ]:
PROBES = {"cause": "q_cause", "effect": "q_eff", "med": "q_med"}
ref = ev[["scenario_id", "item_id"]].copy()
for cond in ["base", "see", "do"]:
    for p, qcol in PROBES.items():
        ref[f"{p}_{cond}"] = belief(ev[cond].tolist(), ev[qcol].tolist(), ev[f"span_{cond}"].tolist())
for cond in ["see", "do"]:
    for p in PROBES:
        ref[f"d{p}_{cond}"] = ref[f"{p}_{cond}"] - ref[f"{p}_base"]
ref.to_csv(f"{OUT}/A_reference_items.csv", index=False)

sc = ref.groupby("scenario_id")[[f"d{p}_{c}" for c in ["see", "do"] for p in PROBES]].mean()
sc["r_see"] = sc.dcause_see / sc.deffect_see
sc["r_do"] = sc.dcause_do / sc.deffect_do
sc["valid"] = ((sc.dcause_see > 0.5) & (sc.deffect_see > 0.5) & (sc.deffect_do > 0.5)
               & (sc.r_see - sc.r_do > 0.25))
sc.to_csv(f"{OUT}/A_reference_scenarios.csv")
display(sc.round(2))
VALID = sc.index[sc.valid].tolist()
print(f"valid scenarios: {len(VALID)} / {len(sc)}")

## Stage B: concept directions + probe check
For each scenario and layer: direction = mean activation when the mediator is **true** (see text) − mean when it is **false** (neg text), at the last token of the mediator sentence, learned on the 16 training variants only.
Probe check: on the held-out variants, does projecting onto the direction tell true from false? Accuracy near 1.0 means the direction really encodes the mediator.

In [ ]:
Htr_t = key_states(train.see.tolist(), train.q_med.tolist(), train.span_see.tolist(), LAYERS)
Htr_f = key_states(train.neg.tolist(), train.q_med.tolist(), train.span_neg.tolist(), LAYERS)
Hev_t = key_states(ev.see.tolist(), ev.q_med.tolist(), ev.span_see.tolist(), LAYERS)
Hev_f = key_states(ev.neg.tolist(), ev.q_med.tolist(), ev.span_neg.tolist(), LAYERS)
Hev_b = key_states(ev.base.tolist(), ev.q_med.tolist(), ev.span_base.tolist(), LAYERS)

DIRS, rows = {}, []
for s in SCEN:
    it, ie = (train.scenario_id == s).values, (ev.scenario_id == s).values
    mt, mf = Htr_t[it].mean(0), Htr_f[it].mean(0)          # [L, H]
    dvec, mid = mt - mf, (mt + mf) / 2
    DIRS[s] = dvec
    for li, L in enumerate(LAYERS):
        pt = ((Hev_t[ie, li] - mid[li]) @ dvec[li] > 0).float()
        pf = ((Hev_f[ie, li] - mid[li]) @ dvec[li] < 0).float()
        pb = ((Hev_b[ie, li] - mid[li]) @ dvec[li]) / dvec[li].norm() ** 2   # where base sits: 0 = midpoint
        rows.append(dict(scenario_id=s, layer=L, probe_acc=torch.cat([pt, pf]).mean().item(),
                         dir_norm=dvec[li].norm().item(), resid_norm=Hev_b[ie, li].norm(dim=-1).mean().item(),
                         base_proj=pb.mean().item()))
probe = pd.DataFrame(rows)
probe.to_csv(f"{OUT}/B_probe.csv", index=False)
torch.save({"layers": LAYERS, "dirs": DIRS}, f"{OUT}/B_directions.pt")
display(probe.groupby("layer")[["probe_acc", "dir_norm", "resid_norm", "base_proj"]].mean().round(3))

## Stage C: the edit sweep
Edit the **base** story (no words changed) by adding α × direction at one layer and one position, then ask the cause, effect and mediator questions.
Three kinds of direction: **real** (this scenario's mediator), **random** (same length), **unrelated** (another scenario's mediator, same length).
Controls run at α = 2 only. Results are saved after every layer; if Colab disconnects, rerun this cell and it skips finished layers.

In [ ]:
EV = ev[ev.scenario_id.isin(VALID)].reset_index(drop=True)
n_items = len(EV)
n_runs = len(LAYERS) * len(POSITIONS) * (len(ALPHAS) + 2) * len(PROBES)
print(f"{n_items} items x {n_runs} settings = {n_items * n_runs:,} forward passes")

gen = torch.Generator().manual_seed(0)
RAND = {s: torch.randn(len(LAYERS), Htr_t.shape[-1], generator=gen) for s in SCEN}
POOL = VALID if len(VALID) > 1 else SCEN          # "unrelated" = the next scenario's mediator direction
OTHER = {s: POOL[(POOL.index(s) + 1) % len(POOL)] for s in POOL}

def vecs_for(kind, li, alpha):
    out = []
    for s in EV.scenario_id:
        real = DIRS[s][li]
        if kind == "real":
            v = real
        elif kind == "random":
            v = RAND[s][li] / RAND[s][li].norm() * real.norm()
        else:
            o = DIRS[OTHER[s]][li]
            v = o / o.norm() * real.norm()
        out.append(alpha * v)
    return torch.stack(out)

stories, spans = EV.base.tolist(), EV.span_base.tolist()
base_b = {p: belief(stories, EV[q].tolist(), spans) for p, q in PROBES.items()}
SETTINGS = [("real", a) for a in ALPHAS] + [("random", 2.0), ("unrelated", 2.0)]
t0 = time.time()
for li, L in enumerate(LAYERS):
    path = f"{OUT}/C_sweep_layer{L:02d}.csv"
    if os.path.exists(path):
        print("skip layer", L); continue
    rows = []
    for mode in POSITIONS:
        for kind, alpha in SETTINGS:
            vecs = vecs_for(kind, li, alpha)
            r = EV[["scenario_id", "item_id"]].copy()
            r["layer"], r["position"], r["kind"], r["alpha"] = L, mode, kind, alpha
            for p, q in PROBES.items():
                r[f"d{p}"] = belief(stories, EV[q].tolist(), spans, edit=dict(layer=L, vecs=vecs, mode=mode)) - base_b[p]
            rows.append(r)
    pd.concat(rows).to_csv(path, index=False)
    print(f"layer {L} done ({(time.time() - t0) / 60:.1f} min elapsed)")

## Stage D: do-likeness score
Per scenario: r = Δcause / Δeffect for see, do and the edit, then **D = (r_see − r_edit) / (r_see − r_do)**.
**D = 0** means the edit acts like *seeing*; **D = 1** means it acts like the model's own *doing*.
An edit only counts if it moved the effect by at least half as much as seeing did (matched-effect rule). Medians across scenarios, with 95% bootstrap intervals that resample scenarios.

In [ ]:
import glob, matplotlib.pyplot as plt
sw = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/C_sweep_layer*.csv")))
g = sw.groupby(["scenario_id", "layer", "position", "kind", "alpha"])[["dcause", "deffect", "dmed"]].mean().reset_index()
g = g.merge(sc[["deffect_see", "r_see", "r_do"]], left_on="scenario_id", right_index=True)
g["works"] = g.deffect >= 0.5 * g.deffect_see
g["r_edit"] = g.dcause / g.deffect
g["D"] = np.where(g.works, (g.r_see - g.r_edit) / (g.r_see - g.r_do), np.nan)
g["D_clip"] = g.D.clip(0, 1)
g.to_csv(f"{OUT}/D_scenario_scores.csv", index=False)

rng = np.random.default_rng(0)
def boot(x, n=2000):
    x = x.dropna().values
    if len(x) < 3:
        return np.nan, np.nan
    m = [np.median(rng.choice(x, len(x))) for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

summ = []
for k, grp in g.groupby(["kind", "alpha", "layer", "position"]):
    lo, hi = boot(grp.D)
    summ.append(dict(zip(["kind", "alpha", "layer", "position"], k), n_scen=len(grp), works_rate=grp.works.mean(),
                     n_scored=grp.D.notna().sum(), D_median=grp.D.median(), D_clip_median=grp.D_clip.median(),
                     ci_lo=lo, ci_hi=hi, dmed_mean=grp.dmed.mean()))
summ = pd.DataFrame(summ)
summ.to_csv(f"{OUT}/D_summary.csv", index=False)

# Heatmaps: one panel per alpha (real direction). Colour = median clipped D; text = median D and n scored.
real = summ[summ.kind == "real"]
fig, axes = plt.subplots(1, len(ALPHAS), figsize=(4.2 * len(ALPHAS), 5), sharey=True)
for ax, a in zip(np.atleast_1d(axes), ALPHAS):
    m = real[real.alpha == a].pivot(index="layer", columns="position", values="D_clip_median").reindex(columns=POSITIONS)
    n = real[real.alpha == a].pivot(index="layer", columns="position", values="n_scored").reindex(columns=POSITIONS)
    im = ax.imshow(m.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = m.values[i, j]
            ax.text(j, i, "–" if np.isnan(v) else f"{v:.2f}\nn={int(n.values[i, j])}", ha="center", va="center",
                    fontsize=8, color="white" if (not np.isnan(v) and v > 0.6) else "black")
    ax.set_xticks(range(len(POSITIONS)), POSITIONS); ax.set_yticks(range(len(m.index)), m.index)
    ax.set_title(f"alpha = {a}"); ax.set_xlabel("edit position")
np.atleast_1d(axes)[0].set_ylabel("layer")
fig.colorbar(im, ax=axes, label="median D (0 = seeing, 1 = doing)")
fig.suptitle("Do-likeness of activation edits (real direction)")
plt.savefig(f"{OUT}/D_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

# Controls: how often does each kind of edit move the effect enough to count?
ctrl = summ[summ.alpha == 2.0].groupby("kind")[["works_rate", "dmed_mean", "D_median"]].mean()
display(ctrl.round(2))
print("Expect: real >> random and unrelated on works_rate and dmed_mean. If not, the directions are not specific.")